text tokenization practice 

In [7]:
import os
import requests 
if not   os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)   #if the verdict dosent exist go to this path and downaload it we are gonna use this short story as our data sample 

to read the text

In [8]:
# Open the file in "read" represnted by r  and save the contents to a variable as f
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
# Verify it worked by checking the length and printing the first 100 characters
print("Total characters:", len(raw_text))
print(raw_text[:100])

Total characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


taking raw text and tokeniazing it

In [9]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

print("\nTotal tokens:", len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']

Total tokens: 4690


In [10]:
# 1. Remove all duplicates using set preprocessed by converting the list to a set, 
 #then sort alphabetically using sorted
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Unique vocabulary size:", vocab_size)
# 2. Build the hash table (dictionary) mapping string tokens to integer IDs
vocab = {token: integer for integer, token in enumerate(all_words)}
# 3. Print a small sample of the dictionary to verify the mapping 
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

Unique vocabulary size: 1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)


In [11]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        # Create a reverse dictionary that maps integers back to strings
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        # Process the input text into tokens just like we did before
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Convert the tokens into integer IDs using our dictionary
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        # Convert integer IDs back to text tokens
        text = " ".join([self.int_to_str[i] for i in ids])
        # Clean up the spaces around punctuation for readability
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text    #this  code combines all the logic from before 

In [12]:
# 1. Instantiate the tokenizer
tokenizer = SimpleTokenizerV1(vocab)

# 2. Define the test text (using 3 quotes)
text = """It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""

# 3. Test the encoder
ids = tokenizer.encode(text)
print("Encoded IDs:\n", ids)

# 4. Test the decoder
decoded_text = tokenizer.decode(ids)
print("\nDecoded Text:\n", decoded_text)

Encoded IDs:
 [56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]

Decoded Text:
 It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [13]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.14.0


In [14]:
tokenizer = tiktoken.get_encoding("gpt2") #this line downloads and loads the exact pre-trained BPE vocabulary and merging rules that OpenAI used for the GPT-2 model

In [15]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"}) #testing the BPE with out of vocab text

print(integers) 

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [16]:
strings = tokenizer.decode(integers)

print(strings) #returning it back to text

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [17]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [18]:
enc_sample = enc_text[50:]

In [19]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [20]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


In [21]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [22]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [23]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [24]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [24]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [25]:
input_ids = torch.tensor([2, 3, 5, 1]) #pytorch tensor 

In [26]:
vocab_size = 6 #Total number of unique words in our dictionary this dictates the number of rows 
output_dim = 3 #The amount of numbers used to represent one single word it dictates the number of columns

torch.manual_seed(123) #telling it where to star from 
embedding_layer = torch.nn.Embedding(vocab_size, output_dim) # printing the table

In [27]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [28]:
print(embedding_layer(torch.tensor([3]))) #telling it which row to grab

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [29]:
print(embedding_layer(input_ids)) # calling the ids in cell 25

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [25]:
vocab_size = 50257  #Total number of unique words in our dictionary this dictates the number of rows
output_dim = 256  #The amount of numbers used to represent one single word it dictates the number of columns

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim) #bulids the matrix 

In [26]:
max_length = 4 # model will only take 4 words at a time
dataloader = create_dataloader_v1(
    raw_text, batch_size=8 , max_length=max_length,
    stride=max_length, shuffle=False # batch size is how many four words will be taken each time at this case its 8                                    
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [27]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [34]:
token_embeddings = token_embedding_layer(inputs)#Takesthe input IDs from above looks them up in the matrix and swaps every single ID for with 256 decimalnumbers
print(token_embeddings.shape)
#print(token_embeddings)

torch.Size([8, 4, 256])


In [35]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim) #bulding a second matrix just for positions
#print(pos_embedding_layer.weight)

In [36]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length)) # torch.arange(4) generates [0, 1, 2, 3] -> embedding layer returns rows 0 to 3 as a [4, 256] tensor
print(pos_embeddings.shape)
#print(pos_embeddings)

torch.Size([4, 256])


In [37]:
input_embeddings = token_embeddings + pos_embeddings # Addsthe[4, 256] position tensor to the [8, 4, 256] word tensor -> combines word meaning + word order into one final vector
print(input_embeddings.shape)
#print(input_embeddings)

torch.Size([8, 4, 256])
